In [89]:
import numpy as np
import torch
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Modelo de GPU: {torch.cuda.get_device_name(0)}")

corpus = pd.read_csv('C:/Users/inqui/OneDrive/Desktop/Clases/26-2/LLM_PROJECT_1/data/processed/infraestructura_limpio.csv',
                     usecols = ['comentario', 'rango_humano'],encoding = 'utf-8-sig')
print(corpus.head())

GPU disponible: False
                                          comentario  rango_humano
0  cuál es el más cercado para rayar el nombre de...           3.0
1  esos baños deberian estar en el metro, no sabe...           2.0
2  ????????????????????????????????????no pues bu...           3.0
3                                los van a abandonar           2.0
4                           Nada los tiene contentos           4.0


In [90]:
analizador = pipeline(
    "sentiment-analysis",
#    model = 'citizenlab/distilbert-base-multilingual-cased-toxicity' #podria funcionar con un tuneo
#    model ="BAAI/bge-reranker-v2-m3" #no sirve para el objetivo
#    model="distilbert-base-uncased-finetuned-sst-2-english" #el que usa el profe. podria funcionar tuneado
    model = 'nlptown/bert-base-multilingual-uncased-sentiment', #podria funcionar con un tuneo, el mas prometedor por ahora    
#    model = "FacebookAI/roberta-large-mnli", #puede prometer
#    model = 'FacebookAI/xlm-roberta-large', #demasiado crudo
#    model = 'cardiffnlp/twitter-roberta-base-sentiment-latest', #podria ser, pero hay que enseñarle español
#    model = 'Bhumika/roberta-base-finetuned-sst2',
    torch_dtype="auto",           # Detecta automáticamente el tipo óptimo
#    device_map="auto"             # Coloca el modelo en GPU si está disponible
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

etiqueta = []
confianza = []

for texto in list(corpus['comentario']):
    try:
        aux = analizador(texto)[0]
        etiqueta.append(aux['label'])
        confianza.append(aux['score'])
    except:
        print(texto, type(texto))

Vamos ahora con el calculo de las metricas del baseline

In [91]:
resultados = pd.DataFrame({'Comentario':list(corpus['comentario']),'Puntuacion Estimada': [float(etiq[0]) for etiq in etiqueta],
                           'Puntuacion Real':corpus['rango_humano'],'Confianza':confianza})

print(resultados[0:10])

#aqui se hace la comparacion promedio de atinados y predichos, matriz de confusion, etc...


                                          Comentario  Puntuacion Estimada  \
0  cuál es el más cercado para rayar el nombre de...                  5.0   
1  esos baños deberian estar en el metro, no sabe...                  1.0   
2  ????????????????????????????????????no pues bu...                  1.0   
3                                los van a abandonar                  1.0   
4                           Nada los tiene contentos                  1.0   
5                                             ??????                  1.0   
6    se ve como 1520 .... no se de que te quejas????                  1.0   
7                                        no maaaa...                  1.0   
8                     y que volteas hacia arriba? ??                  1.0   
9  Jajajaja exacto hay un montón de fugas porque ...                  1.0   

   Puntuacion Real  Confianza  
0              3.0   0.248160  
1              2.0   0.329172  
2              3.0   0.457976  
3              2.0   0.5

In [88]:
#k-cross validation para fine tuning
def kfold_cross_validation(df, k=5, shuffle=True, random_state=None, stratify=False):
    """
    Genera particiones para k-fold cross validation.
    
    Parámetros:
    -----------
    df : DataFrame
        DataFrame a particionar
    k : int
        Número de folds (particiones)
    shuffle : bool
        Si mezclar los datos antes de particionar
    random_state : int
        Semilla para reproducibilidad
    stratify : bool o Series
        Si es True, usa la primera columna como estratificación
        Si es Series, usa esa columna para estratificar
    
    Retorna:
    --------
    list : Lista de tuplas (train_indices, test_indices)
    """
    
    if stratify:
        # Obtener etiquetas para estratificación
        if isinstance(stratify, bool):
            y = df.iloc[:, 1]  # Usa primera columna
        else:
            y = df[stratify]
        kfold = StratifiedKFold(n_splits=k, shuffle=shuffle, random_state=random_state)
    else:
        kfold = KFold(n_splits=k, shuffle=shuffle, random_state=random_state)
    
    # Generar folds
    folds = list(kfold.split(df, y if stratify else None))
    
    return folds

# K-Fold simple
muestra = corpus[np.isnan(corpus['rango_humano']) == False]
semilla = 61298
folds = kfold_cross_validation(muestra, k=5, shuffle=True, random_state=42)

etiqueta_train = []
confianza_train = []

for i, (train_idx, test_idx) in enumerate(folds):
    print(f"Fold {i+1}: Train={len(train_idx)}, Test={len(test_idx)}")
    
    # Obtenemos los DataFrames
    train_df = muestra.iloc[train_idx]
    test_df = muestra.iloc[test_idx]
    

    
    pass


Fold 1: Train=176, Test=44
Fold 2: Train=176, Test=44
Fold 3: Train=176, Test=44
Fold 4: Train=176, Test=44
Fold 5: Train=176, Test=44
